In [1]:
# %%
# Notebook 08: 08_eval_full_dataset.ipynb
# هدف: اجرای کامل گراف روی structured_questions.csv و محاسبه Accuracy کلی و per-category

import os, sys, time, json, traceback
from pathlib import Path

import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent  # اگر داخل notebooks هستی
SRC_PATH = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_PATH))

from dotenv import load_dotenv
load_dotenv()

print(f"✓ PROJECT_ROOT: {PROJECT_ROOT}")
print(f"✓ SRC_PATH: {SRC_PATH}")
print("✓ OPENROUTER_API_KEY:", (os.getenv("OPENROUTER_API_KEY") or "")[:10], "...")
print("✓ COHERE_API_KEY:", (os.getenv("COHERE_API_KEY") or "")[:10], "...")


✓ PROJECT_ROOT: f:\Thesis\project\3-Multi-Agent-System\Test_402\MA\legal_multi_agent
✓ SRC_PATH: f:\Thesis\project\3-Multi-Agent-System\Test_402\MA\legal_multi_agent\src
✓ OPENROUTER_API_KEY: sk-or-v1-4 ...
✓ COHERE_API_KEY: O0pIM8Zkm9 ...


In [2]:
# %%
from legal_multi_agent.graphs.main_graph import graph

print("✓ Graph imported")

✓ Graph imported


In [3]:
# %%
QUESTIONS_PATH = r"F:\Thesis\project\403-vekalat\structured_questions_402.csv"
GOLD_PATH      = r"F:\Thesis\project\3-Multi-Agent-System\Test_402\BaseLine\Gold_402\Gold_402.csv"

df_q = pd.read_csv(QUESTIONS_PATH)
df_gold = pd.read_csv(GOLD_PATH)

print("Questions:", len(df_q))
print("Gold:", len(df_gold))

display(df_q.head(2))
display(df_gold.head(2))


Questions: 10
Gold: 10


,question_number,category,question,options
0,1,نامشخص,«الف» به وکالت از «ب»، ملک «ب» را به مبلغ معین...,1) درهرحالت، «الف» مکلف به استرداد ثمن دریافتی...
1,2,نامشخص,درباره «دلیل لبّی»، کدام مورد صحیح است؟,1) فقط باید اخذ به قدر متیقّن کرد. | 2) اصول ل...


,idx,Gold,explain
0,1,3,براساس ماده 733 قانون مدنی، اگر بیع به واسطه ف...
1,2,1,دلایل لُبّی (مانند عقل، اجماع و سیره عقلا) به ...


In [4]:
# %%
# دیتاست 1402 — 10 سوال (بدون حذف و بدون multi_gold)

def normalize_gold_value(gold_val: str) -> str:
    """نرمال‌سازی مقدار Gold برای دیتاست 1402"""
    val = str(gold_val).strip()
    digits = [ch for ch in val if ch.isdigit()]
    if not digits:
        return val
    if len(digits) == 1:
        return digits[0]
    return ",".join(digits)

df_gold["Gold"] = [
    normalize_gold_value(gold)
    for gold in df_gold["Gold"]
]

print(f"✓ تعداد سوالات Gold: {len(df_gold)}")
display(df_gold.head(10))

✓ تعداد سوالات Gold: 10


,idx,Gold,explain
0,1,3,براساس ماده 733 قانون مدنی، اگر بیع به واسطه ف...
1,2,1,دلایل لُبّی (مانند عقل، اجماع و سیره عقلا) به ...
2,3,2,با لحاظ ماده 3 قانون تشدید مجازات اسیدپاشی و ح...
3,4,1,بر اساس ماده ۲ قانون جرم سیاسی، توهین و افترا ...
4,5,4,در خصوص ابراء عبارت است از: «ابراء کننده می‌تو...
5,6,2,موضوع استرداد دعوی در آیین دادرسی مدنی به دو م...
6,7,2,ابلاغ قانونی: اگرچه اخطاریه به نازنین ابلاغ قا...
7,8,1,این مورد دقیقاً مطابق با ماده ۲۰۹ لایحه اصلاحی...
8,9,4,بر اساس اصل ۱۴۱ قانون اساسی جمهوری اسلامی ایرا...
9,10,2,بر اساس اصل ۱۶۷ قانون اساسی جمهوری اسلامی ایرا...


In [5]:
# %%
def parse_options_to_text(options_raw) -> str:
    """تبدیل ستون options به متن استاندارد برای گراف."""
    if options_raw is None or (isinstance(options_raw, float) and pd.isna(options_raw)):
        return ""
    s = str(options_raw).strip()

    # اگر JSON-لایک بود
    if s.startswith("[") or s.startswith("{"):
        try:
            obj = json.loads(s)
            if isinstance(obj, list):
                lines = []
                for i, item in enumerate(obj, start=1):
                    lines.append(f"{i}) {str(item).strip()}")
                return "\n".join(lines)
        except Exception:
            pass

    return s

def is_correct(pred: str, gold: str) -> bool:
    gold_set = {g.strip() for g in str(gold).split(",") if g.strip()}
    return str(pred).strip() in gold_set


In [6]:
# %%
df = df_q.copy()

# اطمینان از اسم ستون‌ها
assert "question_number" in df.columns
assert "question" in df.columns
assert "options" in df.columns

assert "idx" in df_gold.columns
assert "Gold" in df_gold.columns

df = df.merge(df_gold[["idx", "Gold"]], left_on="question_number", right_on="idx", how="left")
df.drop(columns=["idx"], inplace=True)

print("Merged rows:", len(df))
print("Gold missing:", df["Gold"].isna().mean())
display(df.head(3))


Merged rows: 10
Gold missing: 0.0


,question_number,category,question,options,Gold
0,1,نامشخص,«الف» به وکالت از «ب»، ملک «ب» را به مبلغ معین...,1) درهرحالت، «الف» مکلف به استرداد ثمن دریافتی...,3
1,2,نامشخص,درباره «دلیل لبّی»، کدام مورد صحیح است؟,1) فقط باید اخذ به قدر متیقّن کرد. | 2) اصول ل...,1
2,3,نامشخص,کدام مورد نسبت به جرم اسیدپاشی، ممنوعیت مطلق ج...,1) آزادی مشروط | 2) تخفیف مجازات | 3) تعلیق اج...,2


In [7]:
# %%
USE_OPTION_VERIFIER = True
USE_RETRIEVER_TOOL  = False

MAX_REVISIONS = 5
SLEEP_SECONDS = 0.2

MODEL_NAME = os.getenv("MODEL", "unknown-model").replace("/", "-")
OUT_PATH   = PROJECT_ROOT / "data" / "eval" / f"{MODEL_NAME}.csv"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# ✅ نمایش تنظیمات قبل از اجرا
print(f"{'='*45}")
print(f"🤖 Model             : {os.getenv('MODEL', 'NOT SET')}")
print(f"🔍 Option Verifier   : {USE_OPTION_VERIFIER}")
print(f"📚 Retriever Tool    : {USE_RETRIEVER_TOOL}")
print(f"🔁 Max Revisions     : {MAX_REVISIONS}")
print(f"💾 Output Path       : {OUT_PATH}")
print(f"{'='*45}")

🤖 Model             : google/gemini-2.5-flash
🔍 Option Verifier   : True
📚 Retriever Tool    : False
🔁 Max Revisions     : 5
💾 Output Path       : f:\Thesis\project\3-Multi-Agent-System\Test_402\MA\legal_multi_agent\data\eval\google-gemini-2.5-flash.csv


In [8]:
def run_one(row) -> dict:
    qnum         = int(row["question_number"])
    question     = str(row["question"])
    options_text = parse_options_to_text(row["options"])

    initial_state = {
        "question_number":    qnum,
        "question":           question,
        "options_text":       options_text,
        "max_revisions":      MAX_REVISIONS,
        "use_option_verifier": USE_OPTION_VERIFIER,
        "use_retriever_tool": USE_RETRIEVER_TOOL,
        "messages":           [],
        "tool_results":       {},
        "verifier_output":    None,
        "revision_count":     0,
    }

    t0 = time.time()
    try:
        result  = graph.invoke(initial_state, {"recursion_limit": 60})
        elapsed = time.time() - t0

        final = result.get("final_toon") or {}
        pred  = final.get("answer")
        expl  = final.get("explanation")

        gold = row.get("Gold")
        ok   = is_correct(pred, gold) if (pd.notna(gold) and pred) else None

        tqdm.write(f"Q{qnum}: pred={pred} gold={gold} {'✅' if ok else '❌'}")

        messages = result.get("messages") or []

        # ── messages_json: ذخیره کامل پیام‌ها ─────────────────────────
        messages_json = json.dumps(messages, ensure_ascii=False)

        # ── messages_summary: خلاصه خوانا برای CSV ─────────────────────
        msg_summary_lines = []
        for i, msg in enumerate(messages, 1):
            if not isinstance(msg, dict):
                continue

            meta     = msg.get("metadata") or {}
            agent    = meta.get("agent") or msg.get("name") or msg.get("role") or "?"
            role     = msg.get("role", "?")
            decision = meta.get("decision")
            phase    = meta.get("phase", "")

            # ساخت label کامل
            if role == "tool":
                tool_name = meta.get("tool_name") or msg.get("name") or "tool"
                label = f"[TOOL:{tool_name.upper()}]"
            else:
                label_parts = [agent.upper()]
                if phase:
                    label_parts.append(phase)
                if decision:
                    label_parts.append(decision)
                label = "[" + " | ".join(label_parts) + "]"

            content = str(msg.get("content") or "")[:2000]
            msg_summary_lines.append(f"[{i}] {label}\n{content}")

        messages_summary = "\n\n---\n\n".join(msg_summary_lines)

        # ── critic ──────────────────────────────────────────────────────
        critic_toon    = result.get("critic_toon") or {}
        critic_needs   = critic_toon.get("needs_revision")
        critic_issue   = critic_toon.get("issue")

        # ── verifier ────────────────────────────────────────────────────
        verifier = result.get("verifier_output") or {}
        v_rec    = verifier.get("recommended_answer")

        ctx         = result.get("context") or ""
        rag_results = result.get("rag_results") or []

        return {
            "question_number":       qnum,
            "pred":                  pred,
            "explanation":           expl,
            "gold":                  gold,
            "is_correct":            ok,
            "critic_needs_revision": critic_needs,
            "critic_issue":          critic_issue,
            "revision_count":        result.get("revision_count", 0),
            "total_steps":           result.get("total_steps", 0),
            "verifier_recommended":  v_rec,
            "context_len":           len(ctx),
            "n_docs":                len(rag_results),
            "elapsed_sec":           round(elapsed, 2),
            "messages_json":         messages_json,
            "messages_summary":      messages_summary,
            # raw outputs
            "draft_raw":             result.get("draft_raw", ""),
            "critic_raw":            result.get("critic_raw", ""),
            "error":                 None,
        }

    except Exception as e:
        elapsed = time.time() - t0
        tqdm.write(f"Q{qnum} ERROR: {type(e).__name__}: {str(e)[:200]}")
        return {
            "question_number":       qnum,
            "pred":                  None,
            "explanation":           None,
            "gold":                  row.get("Gold"),
            "is_correct":            None,
            "critic_needs_revision": None,
            "critic_issue":          None,
            "revision_count":        None,
            "total_steps":           None,
            "verifier_recommended":  None,
            "context_len":           None,
            "n_docs":                None,
            "elapsed_sec":           round(elapsed, 2),
            "messages_json":         None,
            "messages_summary":      None,
            "draft_raw":             None,
            "critic_raw":            None,
            "error":                 f"{type(e).__name__}: {e}",
        }

In [ ]:
# %%
done_set = set()
if OUT_PATH.exists():
    df_done  = pd.read_csv(OUT_PATH)
    done_set = set(df_done["question_number"].astype(int).tolist())
    print(f"↩️  Resuming. Already done: {len(done_set)}")

rows_out = []
if OUT_PATH.exists():
    rows_out = df_done.to_dict("records")

total_q    = len(df)
remaining  = total_q - len(done_set)
print(f"📋 کل سوالات: {total_q} | انجام‌شده: {len(done_set)} | باقی‌مانده: {remaining}")
print("─" * 55)

for i, (_, row) in enumerate(tqdm(df.iterrows(), total=total_q, desc="Evaluating"), 1):
    qnum = int(row["question_number"])
    if qnum in done_set:
        continue

    # ✅ نمایش مرحله فعلی قبل از اجرا
    done_so_far = len(done_set)
    tqdm.write(f"\n🔄 مرحله {done_so_far + 1}/{total_q} — سوال شماره {qnum}")
    tqdm.write(f"   سوال: {str(row['question'])[:80]}...")

    r = run_one(row)
    rows_out.append(r)
    done_set.add(qnum)

    # ✅ نمایش نتیجه بعد از اجرا
    status = "✅ صحیح" if r.get("is_correct") else ("⚠️ خطا" if r.get("error") else "❌ غلط")
    tqdm.write(f"   {status} | pred={r.get('pred')} | gold={r.get('gold')} | {r.get('elapsed_sec')}s | rev={r.get('revision_count')}")

    # ذخیره دوره‌ای هر ۱۰ سوال
    if len(rows_out) % 10 == 0:
        pd.DataFrame(rows_out).to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
        tqdm.write(f"💾 ذخیره دوره‌ای: {len(rows_out)} سوال")

    time.sleep(SLEEP_SECONDS)

df_res_v2 = pd.DataFrame(rows_out)
df_res_v2.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

# ✅ خلاصه نهایی
correct  = int(df_res_v2["is_correct"].sum()) if "is_correct" in df_res_v2 else 0
errors   = int(df_res_v2["error"].notna().sum()) if "error" in df_res_v2 else 0
answered = len(df_res_v2) - errors
accuracy = correct / answered * 100 if answered > 0 else 0

print(f"\n{'='*55}")
print(f"✅ ذخیره شد: {OUT_PATH}")
print(f"📊 نتیجه نهایی: {correct}/{answered} صحیح | Accuracy: {accuracy:.1f}%")
if errors:
    print(f"⚠️  خطا: {errors} سوال")
print(f"{'='*55}")

📋 کل سوالات: 10 | انجام‌شده: 0 | باقی‌مانده: 10
───────────────────────────────────────────────────────


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]


🔄 مرحله 1/10 — سوال شماره 1
   سوال: «الف» به وکالت از «ب»، ملک «ب» را به مبلغ معین به «ج» می‌فروشد و ثمن معامله نیز ...
📝 Query: «الف» به وکالت از «ب»، ملک «ب» را به مبلغ معین به «ج» می‌فروشد و ثمن معامله نیز ...
🔍 روش انتخاب شده: Semantic search
🔍 Semantic only: 24 نتیجه
   ✓ Retrieved: 24 documents
🔄 Reranking 24 سند (بدون law+article)...
   ✓ Reranked: top 6 documents


Evaluating:   0%|          | 0/10 [00:25<?, ?it/s]

Q1: pred=3 gold=3 ✅
   ✅ صحیح | pred=3 | gold=3 | 25.92s | rev=0


...


🔄 مرحله 10/10 — سوال شماره 10
   سوال: قاضی پرونده در رأی صادره، به نظریه فقهی استناد می نماید که متناظر با ماده ای از ...
📝 Query: قاضی پرونده در رأی صادره، به نظریه فقهی استناد می نماید که متناظر با ماده ای از ...
🎯 روش انتخاب شده: Metadata-aware
🎯 Metadata: law='قانون اساسی' | اصل 167
🔍 فیلتر قانون + شماره (با Range)...
   ✓ 0 سند (law+principle_number)
🔍 فیلتر فقط قانون...
   ✓ 10 سند (law_only)
🔍 Semantic fallback...
✅ نهایی: 24 نتیجه
   ✓ Retrieved: 24 documents
🔄 Reranking 24 سند (بدون law+article)...
   ✓ Reranked: top 6 documents


Evaluating:  90%|█████████ | 9/10 [03:02<00:17, 17.34s/it]

Q10: pred=3 gold=2 ❌
   ❌ غلط | pred=3 | gold=2 | 17.39s | rev=0
💾 ذخیره دوره‌ای: 10 سوال


Evaluating: 100%|██████████| 10/10 [03:03<00:00, 18.32s/it]


✅ ذخیره شد: f:\Thesis\project\3-Multi-Agent-System\Test_402\MA\legal_multi_agent\data\eval\google-gemini-2.5-flash.csv
📊 نتیجه نهایی: 5/10 صحیح | Accuracy: 50.0%


In [10]:
if "OUTPATH" not in dir():
    MODEL_NAME = (os.getenv("MODEL") or "unknown-model").replace("/", "-")
    OUTPATH = PROJECT_ROOT / "data" / "eval" / f"{MODEL_NAME}.csv"
    OUTPATH.parent.mkdir(parents=True, exist_ok=True)

total   = len(df_res_v2)
correct = int(df_res_v2["is_correct"].sum())
errors  = int(df_res_v2["error"].notna().sum()) if "error" in df_res_v2 else 0
accuracy = correct / (total - errors) * 100 if (total - errors) > 0 else 0

print("=" * 55)
print(f"  آزمون ۱۴۰۲ — {total} سؤال")
print("=" * 55)
print(f"  {correct} / {total - errors}")
print(f"  {errors} خطا")
print(f"  Accuracy: {accuracy:.1f}%")
print("=" * 55)

print(f"\n{'QNum':>7} {'Gold':>7} {'Pred':>7} {'OK':>5} {'Rev':>5} {'Steps':>7} {'Time':>7}")
print("-" * 55)
for _, row in df_res_v2.iterrows():
    ok = "✅" if row["is_correct"] else ("⚠️" if pd.notna(row.get("error")) else "❌")
    print(
        f"{str(row['question_number']):>7}",
        f"{str(row.get('gold', '')):>7}",
        f"{str(row.get('pred', '')):>7}",
        f"{ok:>5}",
        f"{str(row.get('revision_count', '')):>5}",
        f"{str(row.get('total_steps', '')):>7}",
        f"{str(row.get('elapsed_sec', '')):>7}",
    )

# ── ذخیره conversation.txt برای هر سؤال ────────────────────────────
CONV_DIR = OUTPATH.parent / "conversations"
CONV_DIR.mkdir(exist_ok=True)

for _, row in df_res_v2.iterrows():
    qnum = row["question_number"]
    if not row.get("messages_json"):
        continue
    try:
        messages = json.loads(row["messages_json"])
    except Exception:
        continue

    fname = CONV_DIR / f"q{int(qnum):03d}_conversation.txt"
    with open(fname, "w", encoding="utf-8") as f:
        # ── هدر ────────────────────────────────────────────────────────
        f.write("=" * 70 + "\n")
        f.write(f"  سؤال {qnum} | Gold: {row.get('gold')} | Pred: {row.get('pred')}")
        f.write(f"  {'✅' if row.get('is_correct') else '❌'}\n")
        f.write(f"  Revision: {row.get('revision_count')} | Steps: {row.get('total_steps')} | Time: {row.get('elapsed_sec')}s\n")
        f.write("=" * 70 + "\n\n")

        # ── پیام‌های ایجنت‌ها ──────────────────────────────────────────
        for i, msg in enumerate(messages, 1):
            if not isinstance(msg, dict):
                continue

            meta     = msg.get("metadata") or {}
            agent    = meta.get("agent") or msg.get("name") or msg.get("role") or "?"
            role     = msg.get("role", "?")
            decision = meta.get("decision")
            phase    = meta.get("phase", "")
            content  = msg.get("content") or ""

            f.write("─" * 50 + "\n")
            f.write(f"[{i}] Agent: {agent.upper()} | Role: {role}")
            if phase:
                f.write(f" | Phase: {phase}")
            if decision:
                f.write(f" | Decision: {decision}")
            f.write("\n")

            # metadata اختصاصی هر ایجنت
            if agent == "critic":
                f.write(f"  Needs Revision : {meta.get('needs_revision')}\n")
                f.write(f"  Issue          : {meta.get('issue')}\n")
                f.write(f"  Action         : {meta.get('action')}\n")
            elif agent == "supervisor":
                f.write(f"  Next           : {meta.get('next') or decision}\n")
                f.write(f"  Reason         : {meta.get('reason')}\n")
            elif agent == "researcher":
                f.write(f"  Num Docs       : {meta.get('num_docs')}\n")
                f.write(f"  Context Len    : {meta.get('context_len')}\n")
            elif role == "tool":
                tool_name = meta.get("tool_name") or msg.get("name")
                f.write(f"  Tool           : {tool_name}\n")
                f.write(f"  Status         : {meta.get('status')}\n")

            f.write(f"\nContent:\n{content}\n\n")

        # ── خلاصه نهایی ────────────────────────────────────────────────
        f.write("=" * 70 + "\n")
        f.write(f"Gold Explanation:\n{row.get('explanation', '')}\n\n")
        if row.get("draft_raw"):
            f.write(f"--- Draft Raw ---\n{row['draft_raw']}\n\n")
        if row.get("critic_raw"):
            f.write(f"--- Critic Raw ---\n{row['critic_raw']}\n")
        f.write("=" * 70 + "\n")

print(f"\n✅ Conversations saved → {CONV_DIR}")

  آزمون ۱۴۰۲ — 10 سؤال
  5 / 10
  0 خطا
  Accuracy: 50.0%

   QNum    Gold    Pred    OK   Rev   Steps    Time
-------------------------------------------------------
      1       3       3     ✅     0       0   25.92
      2       1       1     ✅     0       0   15.81
      3       2       2     ✅     0       0   16.25
      4       1       1     ✅     0       0   17.51
      5       4       3     ❌     0       0   19.41
      6       2       1     ❌     0       0   18.87
      7       2       1     ❌     0       0   17.03
      8       1       4     ❌     0       0   16.99
      9       4       4     ✅     0       0   15.72
     10       2       3     ❌     0       0   17.39

✅ Conversations saved → f:\Thesis\project\3-Multi-Agent-System\Test_402\MA\legal_multi_agent\data\eval\conversations
